# BITS Pilani — M.Tech (AIML) Machine Learning
## Assignment 2: Telco Customer Churn Classification
**Student**: WILP M.Tech (AIML)  
**Dataset**: Telco Customer Churn (IBM / Kaggle) — 7,043 rows, 19 predictor features  
**Environment**: Executed on BITS Virtual Lab

### 1. Import Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_curve
)

print('Libraries imported.')

### 2. Load Dataset & Data Cleaning

In [ ]:
SrcCsvPath = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
TelcoRawDf = pd.read_csv(SrcCsvPath)
print(f'Dataset shape: {TelcoRawDf.shape[0]} rows, {TelcoRawDf.shape[1]} columns')

# Data cleaning
TelcoCleanDf = TelcoRawDf.copy()
if 'customerID' in TelcoCleanDf.columns:
    TelcoCleanDf.drop(columns=['customerID'], inplace=True)

# Handle blank values in TotalCharges
TelcoCleanDf['TotalCharges'] = pd.to_numeric(
    TelcoCleanDf['TotalCharges'].astype(str).str.strip(),
    errors='coerce'
)
TelcoCleanDf['TotalCharges'] = TelcoCleanDf['TotalCharges'].fillna(0.0)

# Binary target encoding
TargetCol = 'Churn'
TelcoCleanDf[TargetCol] = TelcoCleanDf[TargetCol].map({'Yes': 1, 'No': 0})

NumCols = ['tenure', 'MonthlyCharges', 'TotalCharges']
CatCols = [Col for Col in TelcoCleanDf.columns if Col not in NumCols and Col != TargetCol]

print(f'Numeric features: {NumCols}')
print(f'Categorical features ({len(CatCols)}): {CatCols}')

### 3. Train-Test Split & Preprocessing

In [ ]:
SeedVal = 42
TelcoTrnDf, TelcoTstDf = train_test_split(
    TelcoCleanDf,
    test_size=0.20,
    random_state=SeedVal,
    stratify=TelcoCleanDf[TargetCol]
)

print(f'Train: {len(TelcoTrnDf)} rows | Test: {len(TelcoTstDf)} rows')

# Export test_data.csv for Streamlit app
TelcoTstDf.to_csv('test_data.csv', index=False)

XTrnRaw = TelcoTrnDf.drop(columns=[TargetCol])
YTrn = TelcoTrnDf[TargetCol]

XTstRaw = TelcoTstDf.drop(columns=[TargetCol])
YTst = TelcoTstDf[TargetCol]

# Scaler for numeric, one-hot for categorical
TelcoPrep = ColumnTransformer(
    transformers=[
        ('NumScale', StandardScaler(), NumCols),
        ('CatEnc', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), CatCols)
    ]
)

XTrnProc = TelcoPrep.fit_transform(XTrnRaw)
XTstProc = TelcoPrep.transform(XTstRaw)

os.makedirs('model', exist_ok=True)
joblib.dump(TelcoPrep, 'model/telco_preprocessor.joblib')
print('Preprocessor saved to model/telco_preprocessor.joblib')

### 4. Train Models & Evaluate on Test Set

In [ ]:
TelcoClfDict = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=SeedVal),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, min_samples_split=20, min_samples_leaf=10, random_state=SeedVal),
    'KNN': KNeighborsClassifier(n_neighbors=9, weights='uniform', metric='minkowski'),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, probability=True, random_state=SeedVal)
}

ModelFileMap = {
    'Logistic Regression': 'telco_logreg_model.joblib',
    'Decision Tree': 'telco_dtree_model.joblib',
    'KNN': 'telco_knn_model.joblib',
    'Naive Bayes': 'telco_nb_model.joblib',
    'Support Vector Machine': 'telco_svm_model.joblib'
}

MetricRecords = []

for ModelName, ClfModel in TelcoClfDict.items():
    ClfModel.fit(XTrnProc, YTrn)
    YPred = ClfModel.predict(XTstProc)
    YProb = ClfModel.predict_proba(XTstProc)[:, 1]
    
    AccVal = accuracy_score(YTst, YPred)
    
    # Guard clause: check for multiple target classes
    if len(np.unique(YTst)) > 1:
        AucVal = roc_auc_score(YTst, YProb)
    else:
        AucVal = np.nan
        
    PrecVal = precision_score(YTst, YPred, zero_division=0)
    RecVal = recall_score(YTst, YPred, zero_division=0)
    F1Val = f1_score(YTst, YPred, zero_division=0)
    MccVal = matthews_corrcoef(YTst, YPred)
    
    MetricRecords.append({
        'ML Model Name': ModelName,
        'Accuracy': round(AccVal, 4),
        'AUC': round(AucVal, 4) if not np.isnan(AucVal) else 'N/A',
        'Precision': round(PrecVal, 4),
        'Recall': round(RecVal, 4),
        'F1': round(F1Val, 4),
        'MCC': round(MccVal, 4)
    })
    
    joblib.dump(ClfModel, os.path.join('model', ModelFileMap[ModelName]))

TelcoSummaryDf = pd.DataFrame(MetricRecords)
TelcoSummaryDf.to_csv('model/evaluation_metrics_summary.csv', index=False)
print('=== MODEL COMPARISON TABLE ===')
print(TelcoSummaryDf.to_string(index=False))

### 5. Multi-Model ROC Curves

In [ ]:
if len(np.unique(YTst)) > 1:
    plt.figure(figsize=(9, 5))
    for ModelName, ClfModel in TelcoClfDict.items():
        Probs = ClfModel.predict_proba(XTstProc)[:, 1]
        FprArr, TprArr, _ = roc_curve(YTst, Probs)
        AucScore = roc_auc_score(YTst, Probs)
        plt.plot(FprArr, TprArr, label=f'{ModelName} (AUC = {AucScore:.4f})')
    
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.5000)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - Telco Churn Classifiers')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
else:
    print('ROC curve cannot be plotted for single-class test data.')